# 吉林大学23软件数据挖掘期末作业：天猫复购预测

这是数据挖掘课期末作业整理出来的版本，题目选的是天池日常学习赛「[天猫复购预测-挑战 Baseline](https://tianchi.aliyun.com/competition/entrance/231576/information)」。

简单来说，就是根据用户之前的浏览、加购、购买和收藏记录，预测用户之后会不会再次购买某个商户的商品。

最后提交成绩是 `0.695094883992`，当时排名 `62`。排行榜后面可能还会变化，这里只当作一次提交记录。

这个 notebook 保留的是最后能跑通并提交的流程：读数据、做特征、训练 LightGBM、生成提交文件。运行前需要先从天池下载数据，把 CSV 放到 `data/` 下面，文件名保持天池原始命名。


# 先读数据

数据文件放在 `data/` 目录下，文件名保持天池原始命名。日志表里的 `seller_id` 后面统一改成 `merchant_id`，这样和训练集、测试集的字段名一致。


In [ ]:
import gc

import lightgbm as lgb
import numpy as np
import pandas as pd
from lightgbm import early_stopping, log_evaluation
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold, StratifiedGroupKFold, StratifiedKFold


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
SUBMISSION_DIR = PROJECT_ROOT / "submissions"
SUBMISSION_DIR.mkdir(exist_ok=True)


In [ ]:
tc_ui = {
    "user_id": "int32",
    "age_range": "float32",  # 先按 float 读，后面再压小一点
    "gender": "float32",  # 数据里有缺失，直接读整数会麻烦
}
tc_ul = {
    "user_id": "int32",
    "item_id": "int32",
    "cat_id": "int16",
    "seller_id": "int32",  # 后面统一改名成 merchant_id
    "brand_id": "float32",  # 原数据里有缺失，先按 float 读
    "time_stamp": "int16",
    "action_type": "int8",
}
tc_train = {"user_id": "int32", "merchant_id": "int32", "label": "int8"}
tc_test = {"user_id": "int32", "merchant_id": "int32"}  # 测试集只有待预测主键


In [ ]:
# 先把四张表读进来，日志表顺手去重、改字段名
user_info = pd.read_csv(DATA_DIR / "user_info_format1.csv", dtype=tc_ui).drop_duplicates()
user_log = (
    pd.read_csv(DATA_DIR / "user_log_format1.csv", dtype=tc_ul)
    .drop_duplicates()
    .rename(columns={"seller_id": "merchant_id"})
)
train_data = pd.read_csv(DATA_DIR / "train_format1.csv", dtype=tc_train)
test_data = pd.read_csv(DATA_DIR / "test_format1.csv", dtype=tc_test)

# 这两个字段有缺失，读完后再压成 float16
user_info["age_range"] = pd.to_numeric(user_info["age_range"], errors="coerce").astype("float16")
user_info["gender"] = pd.to_numeric(user_info["gender"], errors="coerce").astype("float16")

user_log["brand_id"] = pd.to_numeric(user_log["brand_id"], errors="coerce").astype("float16")


# 做特征

这部分是主要工作。整体思路比较直接：先做用户、商户各自的统计，再补用户和商户这一对之间的互动信息。


## 商户共现 SVD

把用户访问过、购买过、收藏过的商户当成 token 序列，做成低维向量后，再算用户和商户之间的相似度。


In [ ]:
# 先建一个用户特征表，后面的特征都往这里拼
user_feat = pd.DataFrame(user_log.user_id.drop_duplicates().reset_index(drop=True))
user_feat


In [ ]:
# 把用户访问过的商户当成一串 token，后面拿来做共现
def merchant_cat(x):
    return " ".join(map(str, x))


user_merchant_path = pd.DataFrame(user_log.groupby("user_id")["merchant_id"].apply(merchant_cat))
user_feat = user_feat.merge(user_merchant_path, how="left", on="user_id")
del user_merchant_path
gc.collect()
user_feat


In [ ]:
# 用词频矩阵表示“用户-商户”的访问关系
merchant_cnt = CountVectorizer(
    token_pattern=r"[0-9]+", vocabulary=dict(zip(map(str, list(range(1, 4996))), range(4995)))
)
merchant_cnt_mat = merchant_cnt.fit_transform(user_feat.merchant_id.tolist())
merchant_cnt_mat.shape


In [ ]:
# 用 SVD 得到一组低维表示
svd = TruncatedSVD(n_components=20)
user_mer_feat_mat = svd.fit_transform(merchant_cnt_mat)
svd.explained_variance_ratio_.sum()


In [ ]:
# SVD 的 components 可以当作商户侧向量
def muf_col_rename_f(colname):
    return "merchant_feat" + str(colname + 1)


merchant_feat = pd.DataFrame(svd.components_.T)
merchant_feat = merchant_feat.rename(
    columns=dict(zip(merchant_feat.columns, map(muf_col_rename_f, merchant_feat.columns.tolist())))
)


def mer_user_feat_id_col(merchant_feat, merchant_id):
    for k, v in zip(merchant_id.keys(), merchant_id.values()):
        merchant_feat.loc[v, "merchant_id"] = k


mer_user_feat_id_col(merchant_feat, merchant_cnt.vocabulary_)
merchant_feat["merchant_id"] = merchant_feat.merchant_id.astype("int16")
merchant_feat


In [ ]:
# 用户侧向量拼回 user_feat
def umf_col_rename_f(colname):
    return "user_feat" + str(colname + 1)


user_feat = pd.concat(
    [
        user_feat,
        pd.DataFrame(user_mer_feat_mat).rename(
            columns=dict(zip(range(20), map(umf_col_rename_f, list(range(20)))))
        ),
    ],
    axis=1,
).drop("merchant_id", axis=1)
user_feat


In [ ]:
# 买过的商户单独做一组共现，复购任务里这个信息比较有用
user_feat_buy = pd.DataFrame(user_log.user_id.drop_duplicates().reset_index(drop=True))
user_merchant_path = pd.DataFrame(
    user_log[user_log["action_type"] == 2].groupby("user_id")["merchant_id"].apply(merchant_cat)
)
user_feat_buy = user_feat_buy.merge(user_merchant_path, how="left", on="user_id")
del user_merchant_path
gc.collect()
merchant_cnt = CountVectorizer(
    token_pattern=r"[0-9]+", vocabulary=dict(zip(map(str, list(range(1, 4996))), range(4995)))
)
merchant_cnt_mat = merchant_cnt.fit_transform(user_feat_buy.merchant_id.tolist())
svd = TruncatedSVD(n_components=200)
user_mer_feat_mat = svd.fit_transform(merchant_cnt_mat)
print(svd.explained_variance_ratio_.sum())
merchant_feat_buy = pd.DataFrame(svd.components_.T)
merchant_feat_buy = merchant_feat_buy.rename(
    columns=dict(
        zip(merchant_feat_buy.columns, map(muf_col_rename_f, merchant_feat_buy.columns.tolist()))
    )
)
mer_user_feat_id_col(merchant_feat_buy, merchant_cnt.vocabulary_)
merchant_feat_buy["merchant_id"] = merchant_feat_buy.merchant_id.astype("int16")
user_feat_buy = pd.concat(
    [
        user_feat_buy,
        pd.DataFrame(user_mer_feat_mat).rename(
            columns=dict(zip(range(200), map(umf_col_rename_f, list(range(200)))))
        ),
    ],
    axis=1,
).drop("merchant_id", axis=1)
# 收藏也做一组，维度不用太大
user_feat_clc = pd.DataFrame(user_log.user_id.drop_duplicates().reset_index(drop=True))
user_merchant_path = pd.DataFrame(
    user_log[user_log["action_type"] == 3].groupby("user_id")["merchant_id"].apply(merchant_cat)
)
user_feat_clc = user_feat_clc.merge(user_merchant_path, how="left", on="user_id").fillna("0")
del user_merchant_path
gc.collect()
merchant_cnt = CountVectorizer(
    token_pattern=r"[0-9]+", vocabulary=dict(zip(map(str, list(range(1, 4996))), range(4995)))
)
merchant_cnt_mat = merchant_cnt.fit_transform(user_feat_clc.merchant_id.tolist())
svd = TruncatedSVD(n_components=20)
user_mer_feat_mat = svd.fit_transform(merchant_cnt_mat)
print(svd.explained_variance_ratio_.sum())
merchant_feat_clc = pd.DataFrame(svd.components_.T)
merchant_feat_clc = merchant_feat_clc.rename(
    columns=dict(
        zip(merchant_feat_clc.columns, map(muf_col_rename_f, merchant_feat_clc.columns.tolist()))
    )
)
mer_user_feat_id_col(merchant_feat_clc, merchant_cnt.vocabulary_)
merchant_feat_clc["merchant_id"] = merchant_feat_clc.merchant_id.astype("int16")
user_feat_clc = pd.concat(
    [
        user_feat_clc,
        pd.DataFrame(user_mer_feat_mat).rename(
            columns=dict(zip(range(20), map(umf_col_rename_f, list(range(20)))))
        ),
    ],
    axis=1,
).drop("merchant_id", axis=1)


In [ ]:
def add_dot_feature(df, user_feat, merchant_feat, prefix="view"):
    """按样本里的 user_id / merchant_id 对齐向量，然后算点积。"""
    # 只留下向量列
    uX = user_feat.set_index("user_id").astype("float32")  # 每行是一个用户向量
    mX = merchant_feat.set_index("merchant_id").astype("float32")

    # 按样本顺序取对应向量，没找到就补 0
    u_vals = uX.reindex(df["user_id"]).fillna(0.0).to_numpy()
    m_vals = mX.reindex(df["merchant_id"]).fillna(0.0).to_numpy()

    # 两边维度不一致就直接报错，方便早点发现问题
    if u_vals.shape[1] != m_vals.shape[1]:
        raise ValueError(f"维度不一致: user({u_vals.shape[1]}) vs merchant({m_vals.shape[1]})")

    # 按行算点积，比循环快很多
    df[f"{prefix}_feat_dot"] = np.einsum("ij,ij->i", u_vals, m_vals).astype("float32")
    return df


# 训练集点积特征
add_dot_feature(train_data, user_feat, merchant_feat, prefix="view")
add_dot_feature(train_data, user_feat_buy, merchant_feat_buy, prefix="buy")
add_dot_feature(train_data, user_feat_clc, merchant_feat_clc, prefix="clc")

# 测试集点积特征
add_dot_feature(test_data, user_feat, merchant_feat, prefix="view")
add_dot_feature(test_data, user_feat_buy, merchant_feat_buy, prefix="buy")
add_dot_feature(test_data, user_feat_clc, merchant_feat_clc, prefix="clc")


In [ ]:
def add_cos_feature(df, user_feat, merchant_feat, prefix="view"):
    """按样本对齐用户向量和商户向量，然后算余弦相似度。"""
    # 先去重，防止 set_index 后出现重复索引
    uX = user_feat.drop_duplicates("user_id", keep="last").set_index("user_id").astype("float32")
    mX = merchant_feat.drop_duplicates("merchant_id", keep="last").set_index("merchant_id").astype("float32")

    # 按样本顺序取对应向量，没找到就补 0
    U = uX.reindex(df["user_id"]).fillna(0.0).to_numpy()
    M = mX.reindex(df["merchant_id"]).fillna(0.0).to_numpy()

    if U.shape[1] != M.shape[1]:
        raise ValueError(f"维度不一致: user({U.shape[1]}) vs merchant({M.shape[1]})")

    # 余弦相似度里也会用到点积
    dot = np.einsum("ij,ij->i", U, M).astype("float32")

    # 没有历史向量的样本，余弦值就按 0 处理
    nu = np.linalg.norm(U, axis=1)
    nm = np.linalg.norm(M, axis=1)
    denom = (nu * nm).astype("float32")

    with np.errstate(divide="ignore", invalid="ignore"):
        cos = np.divide(dot, denom, out=np.zeros_like(dot), where=denom != 0)

    df[f"{prefix}_feat_cos"] = cos
    return df


# 训练集余弦特征
add_cos_feature(train_data, user_feat, merchant_feat, prefix="view")
add_cos_feature(train_data, user_feat_buy, merchant_feat_buy, prefix="buy")
add_cos_feature(train_data, user_feat_clc, merchant_feat_clc, prefix="clc")

# 测试集余弦特征
add_cos_feature(test_data, user_feat, merchant_feat, prefix="view")
add_cos_feature(test_data, user_feat_buy, merchant_feat_buy, prefix="buy")
add_cos_feature(test_data, user_feat_clc, merchant_feat_clc, prefix="clc")


In [ ]:
# 把购买、收藏对应的商户向量也拼进 merchant_feat
merchant_feat = merchant_feat.merge(merchant_feat_buy, how="left", on="merchant_id")
merchant_feat = merchant_feat.merge(merchant_feat_clc, how="left", on="merchant_id")
user_feat.drop(["user_feat" + str(i) for i in range(1, 21)], axis=1, inplace=True)


## 用户侧统计

这里主要看一个用户整体活跃不活跃、买过哪些东西、离双十一有多近这些信息。


In [ ]:
# 先把性别、年龄这类基础信息并进来
def safe_left_merge(left, right, on, keep_cols=None, suffix="_r"):
    # 只拿这次 merge 用得到的列
    if keep_cols is not None:
        right = right[[on] + keep_cols].drop_duplicates(on)
    else:
        right = right.drop_duplicates(on)

    # 如果之前跑过同名列，先清掉
    overlap = set(right.columns) - {on}
    drop_in_left = []
    for base in overlap:
        for cand in (base, f"{base}_x", f"{base}_y", f"{base}{suffix}"):
            if cand in left.columns:
                drop_in_left.append(cand)
    if drop_in_left:
        left = left.drop(columns=list(set(drop_in_left)))

    # suffix 固定下来，重复运行时更好排查
    out = left.merge(right, on=on, how="left", suffixes=("", suffix))
    return out


# 合并用户基础信息
user_feat = safe_left_merge(
    user_feat, user_info, on="user_id", keep_cols=["age_range", "gender"], suffix="_ui"
)

# 下面主要是用户的行为次数、去重数和时间分布
user_feat = user_feat.merge(
    user_log.groupby("user_id")["merchant_id"].aggregate("count"), how="left", on="user_id"
).rename(columns={"merchant_id": "mer_view_cnt"})
user_feat = user_feat.merge(
    user_log[user_log["action_type"] == 1].groupby("user_id")["action_type"].aggregate("count"),
    how="left",
    on="user_id",
).rename(columns={"action_type": "act1_cnt"})
user_feat = user_feat.merge(
    user_log[user_log["action_type"] == 2].groupby("user_id")["action_type"].aggregate("count"),
    how="left",
    on="user_id",
).rename(columns={"action_type": "act2_cnt"})
user_feat = user_feat.merge(
    user_log[user_log["action_type"] == 3].groupby("user_id")["action_type"].aggregate("count"),
    how="left",
    on="user_id",
).rename(columns={"action_type": "act3_cnt"})
user_feat["act1_cnt"] = user_feat["act1_cnt"].fillna(0).astype("int32")
user_feat["act2_cnt"] = user_feat["act2_cnt"].fillna(0).astype("int32")
user_feat["act3_cnt"] = user_feat["act3_cnt"].fillna(0).astype("int32")
user_feat["act2_ratio"] = user_feat["act2_cnt"] / user_feat["mer_view_cnt"]
user_feat = user_feat.merge(
    user_log[["user_id", "merchant_id"]]
    .drop_duplicates()
    .groupby("user_id")["merchant_id"]
    .aggregate("count"),
    how="left",
    on="user_id",
).rename(columns={"merchant_id": "unimer_view_cnt"})
user_feat["uni_mer_view_ratio"] = user_feat["unimer_view_cnt"] / user_feat["mer_view_cnt"]
user_feat = user_feat.merge(
    user_log[["user_id", "cat_id"]]
    .drop_duplicates()
    .groupby("user_id")["cat_id"]
    .aggregate("count"),
    how="left",
    on="user_id",
).rename(columns={"cat_id": "unicat_view_cnt"})
user_feat = user_feat.merge(
    user_log[["user_id", "item_id"]]
    .drop_duplicates()
    .groupby("user_id")["item_id"]
    .aggregate("count"),
    how="left",
    on="user_id",
).rename(columns={"item_id": "uniitem_view_cnt"})
user_feat = user_feat.merge(
    user_log[["user_id", "brand_id"]]
    .drop_duplicates()
    .groupby("user_id")["brand_id"]
    .aggregate("count"),
    how="left",
    on="user_id",
).rename(columns={"brand_id": "unibrand_view_cnt"})
user_feat["uni_cat_view_ratio"] = user_feat["unicat_view_cnt"] / user_feat["mer_view_cnt"]
user_feat["uni_item_view_ratio"] = user_feat["uniitem_view_cnt"] / user_feat["mer_view_cnt"]
user_feat["uni_brand_view_ratio"] = user_feat["unibrand_view_cnt"] / user_feat["mer_view_cnt"]
user_feat = user_feat.merge(
    user_log.groupby("user_id")["time_stamp"].aggregate("min"), how="left", on="user_id"
).rename(columns={"time_stamp": "view_first_time"})
user_feat = user_feat.merge(
    user_log.groupby("user_id")["time_stamp"].aggregate("max"), how="left", on="user_id"
).rename(columns={"time_stamp": "view_last_time"})
user_feat = user_feat.merge(
    user_log.groupby("user_id")["time_stamp"].aggregate("std"), how="left", on="user_id"
).rename(columns={"time_stamp": "view_time_std"})
user_feat = user_feat.merge(
    user_log.loc[user_log["action_type"] == 2, ["user_id", "merchant_id"]]
    .drop_duplicates()
    .groupby("user_id")["merchant_id"]
    .aggregate("count"),
    how="left",
    on="user_id",
).rename(columns={"merchant_id": "unimer_buy_cnt"})
user_feat = user_feat.merge(
    user_log.loc[user_log["action_type"] == 2, ["user_id", "cat_id"]]
    .drop_duplicates()
    .groupby("user_id")["cat_id"]
    .aggregate("count"),
    how="left",
    on="user_id",
).rename(columns={"cat_id": "unicat_buy_cnt"})
user_feat = user_feat.merge(
    user_log.loc[user_log["action_type"] == 2, ["user_id", "item_id"]]
    .drop_duplicates()
    .groupby("user_id")["item_id"]
    .aggregate("count"),
    how="left",
    on="user_id",
).rename(columns={"item_id": "uniitem_buy_cnt"})
user_feat = user_feat.merge(
    user_log.loc[user_log["action_type"] == 2, ["user_id", "brand_id"]]
    .drop_duplicates()
    .groupby("user_id")["brand_id"]
    .aggregate("count"),
    how="left",
    on="user_id",
).rename(columns={"brand_id": "unibrand_buy_cnt"})
user_feat = user_feat.merge(
    user_log[user_log["action_type"] == 2].groupby("user_id")["time_stamp"].aggregate("min"),
    how="left",
    on="user_id",
).rename(columns={"time_stamp": "buy_first_time"})
user_feat = user_feat.merge(
    user_log[user_log["action_type"] == 2].groupby("user_id")["time_stamp"].aggregate("max"),
    how="left",
    on="user_id",
).rename(columns={"time_stamp": "buy_last_time"})
user_feat = user_feat.merge(
    user_log[user_log["action_type"] == 2].groupby("user_id")["time_stamp"].aggregate("std"),
    how="left",
    on="user_id",
).rename(columns={"time_stamp": "buy_time_std"})
# 统计用户加购行为的去重数和时间分布
user_feat = user_feat.merge(
    user_log.loc[user_log["action_type"] == 1, ["user_id", "merchant_id"]]
    .drop_duplicates()
    .groupby("user_id")["merchant_id"]
    .aggregate("count"),
    how="left",
    on="user_id",
).rename(columns={"merchant_id": "unimer_car_cnt"})
user_feat = user_feat.merge(
    user_log.loc[user_log["action_type"] == 1, ["user_id", "cat_id"]]
    .drop_duplicates()
    .groupby("user_id")["cat_id"]
    .aggregate("count"),
    how="left",
    on="user_id",
).rename(columns={"cat_id": "unicat_car_cnt"})
user_feat = user_feat.merge(
    user_log.loc[user_log["action_type"] == 1, ["user_id", "item_id"]]
    .drop_duplicates()
    .groupby("user_id")["item_id"]
    .aggregate("count"),
    how="left",
    on="user_id",
).rename(columns={"item_id": "uniitem_car_cnt"})
user_feat = user_feat.merge(
    user_log.loc[user_log["action_type"] == 1, ["user_id", "brand_id"]]
    .drop_duplicates()
    .groupby("user_id")["brand_id"]
    .aggregate("count"),
    how="left",
    on="user_id",
).rename(columns={"brand_id": "unibrand_car_cnt"})
user_feat = user_feat.merge(
    user_log[user_log["action_type"] == 1].groupby("user_id")["time_stamp"].aggregate("min"),
    how="left",
    on="user_id",
).rename(columns={"time_stamp": "car_first_time"})
user_feat = user_feat.merge(
    user_log[user_log["action_type"] == 1].groupby("user_id")["time_stamp"].aggregate("max"),
    how="left",
    on="user_id",
).rename(columns={"time_stamp": "car_last_time"})
user_feat = user_feat.merge(
    user_log[user_log["action_type"] == 1].groupby("user_id")["time_stamp"].aggregate("std"),
    how="left",
    on="user_id",
).rename(columns={"time_stamp": "car_time_std"})
# 统计用户收藏行为的去重数和时间分布
user_feat = user_feat.merge(
    user_log.loc[user_log["action_type"] == 3, ["user_id", "merchant_id"]]
    .drop_duplicates()
    .groupby("user_id")["merchant_id"]
    .aggregate("count"),
    how="left",
    on="user_id",
).rename(columns={"merchant_id": "unimer_col_cnt"})
user_feat = user_feat.merge(
    user_log.loc[user_log["action_type"] == 3, ["user_id", "cat_id"]]
    .drop_duplicates()
    .groupby("user_id")["cat_id"]
    .aggregate("count"),
    how="left",
    on="user_id",
).rename(columns={"cat_id": "unicat_col_cnt"})
user_feat = user_feat.merge(
    user_log.loc[user_log["action_type"] == 3, ["user_id", "item_id"]]
    .drop_duplicates()
    .groupby("user_id")["item_id"]
    .aggregate("count"),
    how="left",
    on="user_id",
).rename(columns={"item_id": "uniitem_col_cnt"})
user_feat = user_feat.merge(
    user_log.loc[user_log["action_type"] == 3, ["user_id", "brand_id"]]
    .drop_duplicates()
    .groupby("user_id")["brand_id"]
    .aggregate("count"),
    how="left",
    on="user_id",
).rename(columns={"brand_id": "unibrand_col_cnt"})
user_feat = user_feat.merge(
    user_log[user_log["action_type"] == 3].groupby("user_id")["time_stamp"].aggregate("min"),
    how="left",
    on="user_id",
).rename(columns={"time_stamp": "col_first_time"})
user_feat = user_feat.merge(
    user_log[user_log["action_type"] == 3].groupby("user_id")["time_stamp"].aggregate("max"),
    how="left",
    on="user_id",
).rename(columns={"time_stamp": "col_last_time"})
user_feat = user_feat.merge(
    user_log[user_log["action_type"] == 3].groupby("user_id")["time_stamp"].aggregate("std"),
    how="left",
    on="user_id",
).rename(columns={"time_stamp": "col_time_std"})


## 商户侧统计

商户这边也做一套类似统计，重点是规模、覆盖面、购买和收藏行为的分布。


In [ ]:
merchant_feat["merchant_id"] = merchant_feat["merchant_id"].astype("int32")
# 商户侧主要看规模、覆盖范围和时间分布
merchant_feat = merchant_feat.merge(
    user_log.groupby("merchant_id")["user_id"].aggregate("count"), how="left", on="merchant_id"
).rename(columns={"user_id": "user_view_cnt"})
merchant_feat = merchant_feat.merge(
    user_log[user_log["action_type"] == 2].groupby("merchant_id")["action_type"].aggregate("count"),
    how="left",
    on="merchant_id",
).rename(columns={"action_type": "act2_cnt"})
merchant_feat = merchant_feat.merge(
    user_log[user_log["action_type"] == 3].groupby("merchant_id")["action_type"].aggregate("count"),
    how="left",
    on="merchant_id",
).rename(columns={"action_type": "act3_cnt"})
merchant_feat = merchant_feat.merge(
    user_log[user_log["action_type"] == 1].groupby("merchant_id")["action_type"].aggregate("count"),
    how="left",
    on="merchant_id",
).rename(columns={"action_type": "act1_cnt"})
merchant_feat = merchant_feat.merge(
    user_log[["user_id", "merchant_id"]]
    .drop_duplicates()
    .groupby("merchant_id")["user_id"]
    .aggregate("count"),
    how="left",
    on="merchant_id",
).rename(columns={"user_id": "uniuser_view_cnt"})
merchant_feat = merchant_feat.merge(
    user_log[["cat_id", "merchant_id"]]
    .drop_duplicates()
    .groupby("merchant_id")["cat_id"]
    .aggregate("count"),
    how="left",
    on="merchant_id",
).rename(columns={"cat_id": "unicat_view_cnt"})
merchant_feat = merchant_feat.merge(
    user_log[["item_id", "merchant_id"]]
    .drop_duplicates()
    .groupby("merchant_id")["item_id"]
    .aggregate("count"),
    how="left",
    on="merchant_id",
).rename(columns={"item_id": "uniitem_view_cnt"})
merchant_feat = merchant_feat.merge(
    user_log[["brand_id", "merchant_id"]]
    .drop_duplicates()
    .groupby("merchant_id")["brand_id"]
    .aggregate("count"),
    how="left",
    on="merchant_id",
).rename(columns={"brand_id": "unibrand_view_cnt"})
merchant_feat = merchant_feat.merge(
    user_log.groupby("merchant_id")["time_stamp"].aggregate("min"), how="left", on="merchant_id"
).rename(columns={"time_stamp": "view_first_time"})
merchant_feat = merchant_feat.merge(
    user_log.groupby("merchant_id")["time_stamp"].aggregate("max"), how="left", on="merchant_id"
).rename(columns={"time_stamp": "view_last_time"})
merchant_feat = merchant_feat.merge(
    user_log.groupby("merchant_id")["time_stamp"].aggregate("std"), how="left", on="merchant_id"
).rename(columns={"time_stamp": "view_time_std"})
# 购买行为单独统计一套
merchant_feat = merchant_feat.merge(
    user_log.loc[user_log["action_type"] == 2, ["user_id", "merchant_id"]]
    .drop_duplicates()
    .groupby("merchant_id")["user_id"]
    .aggregate("count"),
    how="left",
    on="merchant_id",
).rename(columns={"user_id": "uniuser_buy_cnt"})
merchant_feat = merchant_feat.merge(
    user_log.loc[user_log["action_type"] == 2, ["cat_id", "merchant_id"]]
    .drop_duplicates()
    .groupby("merchant_id")["cat_id"]
    .aggregate("count"),
    how="left",
    on="merchant_id",
).rename(columns={"cat_id": "unicat_buy_cnt"})
merchant_feat = merchant_feat.merge(
    user_log.loc[user_log["action_type"] == 2, ["item_id", "merchant_id"]]
    .drop_duplicates()
    .groupby("merchant_id")["item_id"]
    .aggregate("count"),
    how="left",
    on="merchant_id",
).rename(columns={"item_id": "uniitem_buy_cnt"})
merchant_feat = merchant_feat.merge(
    user_log.loc[user_log["action_type"] == 2, ["brand_id", "merchant_id"]]
    .drop_duplicates()
    .groupby("merchant_id")["brand_id"]
    .aggregate("count"),
    how="left",
    on="merchant_id",
).rename(columns={"brand_id": "unibrand_buy_cnt"})
merchant_feat = merchant_feat.merge(
    user_log[user_log["action_type"] == 2].groupby("merchant_id")["time_stamp"].aggregate("min"),
    how="left",
    on="merchant_id",
).rename(columns={"time_stamp": "buy_first_time"})
merchant_feat = merchant_feat.merge(
    user_log[user_log["action_type"] == 2].groupby("merchant_id")["time_stamp"].aggregate("max"),
    how="left",
    on="merchant_id",
).rename(columns={"time_stamp": "buy_last_time"})
merchant_feat = merchant_feat.merge(
    user_log[user_log["action_type"] == 2].groupby("merchant_id")["time_stamp"].aggregate("std"),
    how="left",
    on="merchant_id",
).rename(columns={"time_stamp": "buy_time_std"})
# 收藏行为也单独统计一套
merchant_feat = merchant_feat.merge(
    user_log.loc[user_log["action_type"] == 3, ["user_id", "merchant_id"]]
    .drop_duplicates()
    .groupby("merchant_id")["user_id"]
    .aggregate("count"),
    how="left",
    on="merchant_id",
).rename(columns={"user_id": "uniuser_col_cnt"})
merchant_feat = merchant_feat.merge(
    user_log.loc[user_log["action_type"] == 3, ["cat_id", "merchant_id"]]
    .drop_duplicates()
    .groupby("merchant_id")["cat_id"]
    .aggregate("count"),
    how="left",
    on="merchant_id",
).rename(columns={"cat_id": "unicat_col_cnt"})
merchant_feat = merchant_feat.merge(
    user_log.loc[user_log["action_type"] == 3, ["item_id", "merchant_id"]]
    .drop_duplicates()
    .groupby("merchant_id")["item_id"]
    .aggregate("count"),
    how="left",
    on="merchant_id",
).rename(columns={"item_id": "uniitem_col_cnt"})
merchant_feat = merchant_feat.merge(
    user_log.loc[user_log["action_type"] == 3, ["brand_id", "merchant_id"]]
    .drop_duplicates()
    .groupby("merchant_id")["brand_id"]
    .aggregate("count"),
    how="left",
    on="merchant_id",
).rename(columns={"brand_id": "unibrand_col_cnt"})
merchant_feat = merchant_feat.merge(
    user_log[user_log["action_type"] == 3].groupby("merchant_id")["time_stamp"].aggregate("min"),
    how="left",
    on="merchant_id",
).rename(columns={"time_stamp": "col_first_time"})
merchant_feat = merchant_feat.merge(
    user_log[user_log["action_type"] == 3].groupby("merchant_id")["time_stamp"].aggregate("max"),
    how="left",
    on="merchant_id",
).rename(columns={"time_stamp": "col_last_time"})
merchant_feat = merchant_feat.merge(
    user_log[user_log["action_type"] == 3].groupby("merchant_id")["time_stamp"].aggregate("std"),
    how="left",
    on="merchant_id",
).rename(columns={"time_stamp": "col_time_std"})
# 加购行为也补一套
merchant_feat = merchant_feat.merge(
    user_log.loc[user_log["action_type"] == 1, ["user_id", "merchant_id"]]
    .drop_duplicates()
    .groupby("merchant_id")["user_id"]
    .aggregate("count"),
    how="left",
    on="merchant_id",
).rename(columns={"user_id": "uniuser_car_cnt"})
merchant_feat = merchant_feat.merge(
    user_log.loc[user_log["action_type"] == 1, ["cat_id", "merchant_id"]]
    .drop_duplicates()
    .groupby("merchant_id")["cat_id"]
    .aggregate("count"),
    how="left",
    on="merchant_id",
).rename(columns={"cat_id": "unicat_car_cnt"})
merchant_feat = merchant_feat.merge(
    user_log.loc[user_log["action_type"] == 1, ["item_id", "merchant_id"]]
    .drop_duplicates()
    .groupby("merchant_id")["item_id"]
    .aggregate("count"),
    how="left",
    on="merchant_id",
).rename(columns={"item_id": "uniitem_car_cnt"})
merchant_feat = merchant_feat.merge(
    user_log.loc[user_log["action_type"] == 1, ["brand_id", "merchant_id"]]
    .drop_duplicates()
    .groupby("merchant_id")["brand_id"]
    .aggregate("count"),
    how="left",
    on="merchant_id",
).rename(columns={"brand_id": "unibrand_car_cnt"})
merchant_feat = merchant_feat.merge(
    user_log[user_log["action_type"] == 1].groupby("merchant_id")["time_stamp"].aggregate("min"),
    how="left",
    on="merchant_id",
).rename(columns={"time_stamp": "car_first_time"})
merchant_feat = merchant_feat.merge(
    user_log[user_log["action_type"] == 1].groupby("merchant_id")["time_stamp"].aggregate("max"),
    how="left",
    on="merchant_id",
).rename(columns={"time_stamp": "car_last_time"})
merchant_feat = merchant_feat.merge(
    user_log[user_log["action_type"] == 1].groupby("merchant_id")["time_stamp"].aggregate("std"),
    how="left",
    on="merchant_id",
).rename(columns={"time_stamp": "car_time_std"})


In [ ]:
# 单独算一下“这个人/这个商户有没有复购痕迹”
buy = user_log[user_log["action_type"] == 2].reset_index(drop=True)
buy["buycnt"] = buy.groupby(["user_id", "merchant_id"])["merchant_id"].transform("count")
buy["if_rebuy"] = buy["buycnt"].map(lambda x: 0 if x <= 1 else 1)
# 用户复购过几个商户
user_feat = user_feat.merge(
    buy.loc[buy["if_rebuy"] == 1, ["user_id", "merchant_id"]]
    .drop_duplicates()
    .groupby("user_id")["merchant_id"]
    .aggregate("count"),
    how="left",
    on="user_id",
).rename(columns={"merchant_id": "rebuy_mer_cnt"})
user_feat["rebuy_mer_cnt"] = user_feat["rebuy_mer_cnt"].fillna(0).astype("int32")
# 双十一前最后一次购买时间，新用户这里会空着
user_feat = user_feat.merge(
    buy[buy["time_stamp"] != 1111]
    .groupby("user_id")["time_stamp"]
    .aggregate("max")
    .rename("befor1111_last_time"),
    how="left",
    on="user_id",
)


# 一个比较粗的购买频率
def time_freq(x):
    if x.max() - x.min():
        return len(x) / (x.max() - x.min())
    else:
        if len(x) > 2:
            return len(x)
        else:
            return 0.0


user_feat = user_feat.merge(
    buy.groupby("user_id")["time_stamp"].aggregate(time_freq).rename("time_freq"),
    how="left",
    on="user_id",
)
# 商户被多少用户复购过
merchant_feat = merchant_feat.merge(
    buy.loc[buy["if_rebuy"] == 1, ["user_id", "merchant_id"]]
    .drop_duplicates()
    .groupby("merchant_id")["user_id"]
    .aggregate("count"),
    how="left",
    on="merchant_id",
).rename(columns={"user_id": "rebuy_user_cnt"})
merchant_feat["rebuy_user_cnt"] = merchant_feat["rebuy_user_cnt"].fillna(0).astype("int32")


In [ ]:
# 把前面做好的用户/商户特征拼到训练集和测试集
# 先把主键类型对齐，不然后面 merge 容易踩坑
for df in (train_data, test_data):
    df["user_id"] = df["user_id"].astype("int32")
    df["merchant_id"] = df["merchant_id"].astype("int32")

user_feat = user_feat.drop_duplicates("user_id", keep="last").copy()
merchant_feat = merchant_feat.drop_duplicates("merchant_id", keep="last").copy()
user_feat["user_id"] = user_feat["user_id"].astype("int32")
merchant_feat["merchant_id"] = merchant_feat["merchant_id"].astype("int32")

# 加前缀，防止用户侧和商户侧同名字段撞在一起
u_cols = [c for c in user_feat.columns if c != "user_id"]
m_cols = [c for c in merchant_feat.columns if c != "merchant_id"]
user_feat = user_feat[["user_id"] + u_cols].rename(columns={c: f"u_{c}" for c in u_cols})
merchant_feat = merchant_feat[["merchant_id"] + m_cols].rename(
    columns={c: f"m_{c}" for c in m_cols}
)

# 合并到样本表
train_data = train_data.merge(user_feat, how="left", on="user_id").merge(
    merchant_feat, how="left", on="merchant_id"
)

test_data = test_data.merge(user_feat, how="left", on="user_id").merge(
    merchant_feat, how="left", on="merchant_id"
)

# 统计不到的地方按 0 处理
num_cols_tr = train_data.select_dtypes(include="number").columns
num_cols_te = test_data.select_dtypes(include="number").columns
train_data[num_cols_tr] = train_data[num_cols_tr].fillna(0)
test_data[num_cols_te] = test_data[num_cols_te].fillna(0)


In [ ]:
# 样本对本身的交互次数也很关键
train_data = train_data.merge(
    user_log.loc[
        (user_log["time_stamp"] == 1111) & (user_log["action_type"] == 2),
        ["user_id", "merchant_id"],
    ]
    .groupby(["user_id", "merchant_id"])["merchant_id"]
    .aggregate("count")
    .rename("1111buy"),
    how="left",
    on=["user_id", "merchant_id"],
)
train_data = train_data.merge(
    user_log.loc[
        (user_log["time_stamp"] == 1111) & (user_log["action_type"] == 3),
        ["user_id", "merchant_id"],
    ]
    .groupby(["user_id", "merchant_id"])["merchant_id"]
    .aggregate("count")
    .rename("1111clc"),
    how="left",
    on=["user_id", "merchant_id"],
)
train_data = train_data.merge(
    user_log.loc[
        (user_log["time_stamp"] == 1111) & (user_log["action_type"] == 0),
        ["user_id", "merchant_id"],
    ]
    .groupby(["user_id", "merchant_id"])["merchant_id"]
    .aggregate("count")
    .rename("1111view"),
    how="left",
    on=["user_id", "merchant_id"],
)
train_data = train_data.merge(
    user_log.loc[
        (user_log["time_stamp"] == 1111) & (user_log["action_type"] == 1),
        ["user_id", "merchant_id"],
    ]
    .groupby(["user_id", "merchant_id"])["merchant_id"]
    .aggregate("count")
    .rename("1111car"),
    how="left",
    on=["user_id", "merchant_id"],
)
test_data = test_data.merge(
    user_log.loc[
        (user_log["time_stamp"] == 1111) & (user_log["action_type"] == 2),
        ["user_id", "merchant_id"],
    ]
    .groupby(["user_id", "merchant_id"])["merchant_id"]
    .aggregate("count")
    .rename("1111buy"),
    how="left",
    on=["user_id", "merchant_id"],
)
test_data = test_data.merge(
    user_log.loc[
        (user_log["time_stamp"] == 1111) & (user_log["action_type"] == 3),
        ["user_id", "merchant_id"],
    ]
    .groupby(["user_id", "merchant_id"])["merchant_id"]
    .aggregate("count")
    .rename("1111clc"),
    how="left",
    on=["user_id", "merchant_id"],
)
test_data = test_data.merge(
    user_log.loc[
        (user_log["time_stamp"] == 1111) & (user_log["action_type"] == 0),
        ["user_id", "merchant_id"],
    ]
    .groupby(["user_id", "merchant_id"])["merchant_id"]
    .aggregate("count")
    .rename("1111view"),
    how="left",
    on=["user_id", "merchant_id"],
)
test_data = test_data.merge(
    user_log.loc[
        (user_log["time_stamp"] == 1111) & (user_log["action_type"] == 1),
        ["user_id", "merchant_id"],
    ]
    .groupby(["user_id", "merchant_id"])["merchant_id"]
    .aggregate("count")
    .rename("1111car"),
    how="left",
    on=["user_id", "merchant_id"],
)
# 训练集：双十一之前的交互
train_data = train_data.merge(
    user_log.loc[
        (user_log["time_stamp"] != 1111) & (user_log["action_type"] == 2),
        ["user_id", "merchant_id"],
    ]
    .groupby(["user_id", "merchant_id"])["merchant_id"]
    .aggregate("count")
    .rename("before1111buy"),
    how="left",
    on=["user_id", "merchant_id"],
)
train_data = train_data.merge(
    user_log.loc[
        (user_log["time_stamp"] != 1111) & (user_log["action_type"] == 3),
        ["user_id", "merchant_id"],
    ]
    .groupby(["user_id", "merchant_id"])["merchant_id"]
    .aggregate("count")
    .rename("before1111clc"),
    how="left",
    on=["user_id", "merchant_id"],
)
train_data = train_data.merge(
    user_log.loc[
        (user_log["time_stamp"] != 1111) & (user_log["action_type"] == 0),
        ["user_id", "merchant_id"],
    ]
    .groupby(["user_id", "merchant_id"])["merchant_id"]
    .aggregate("count")
    .rename("before1111view"),
    how="left",
    on=["user_id", "merchant_id"],
)
train_data = train_data.merge(
    user_log.loc[
        (user_log["time_stamp"] != 1111) & (user_log["action_type"] == 1),
        ["user_id", "merchant_id"],
    ]
    .groupby(["user_id", "merchant_id"])["merchant_id"]
    .aggregate("count")
    .rename("before1111car"),
    how="left",
    on=["user_id", "merchant_id"],
)
test_data = test_data.merge(
    user_log.loc[
        (user_log["time_stamp"] != 1111) & (user_log["action_type"] == 2),
        ["user_id", "merchant_id"],
    ]
    .groupby(["user_id", "merchant_id"])["merchant_id"]
    .aggregate("count")
    .rename("before1111buy"),
    how="left",
    on=["user_id", "merchant_id"],
)
test_data = test_data.merge(
    user_log.loc[
        (user_log["time_stamp"] != 1111) & (user_log["action_type"] == 3),
        ["user_id", "merchant_id"],
    ]
    .groupby(["user_id", "merchant_id"])["merchant_id"]
    .aggregate("count")
    .rename("before1111clc"),
    how="left",
    on=["user_id", "merchant_id"],
)
test_data = test_data.merge(
    user_log.loc[
        (user_log["time_stamp"] != 1111) & (user_log["action_type"] == 0),
        ["user_id", "merchant_id"],
    ]
    .groupby(["user_id", "merchant_id"])["merchant_id"]
    .aggregate("count")
    .rename("before1111view"),
    how="left",
    on=["user_id", "merchant_id"],
)
test_data = test_data.merge(
    user_log.loc[
        (user_log["time_stamp"] != 1111) & (user_log["action_type"] == 1),
        ["user_id", "merchant_id"],
    ]
    .groupby(["user_id", "merchant_id"])["merchant_id"]
    .aggregate("count")
    .rename("before1111car"),
    how="left",
    on=["user_id", "merchant_id"],
)
# 测试集：同样补上历史交互
train_data = train_data.merge(
    user_log[["user_id", "merchant_id", "cat_id"]]
    .drop_duplicates()
    .groupby(["user_id", "merchant_id"])["cat_id"]
    .aggregate("count")
    .rename("um_cat_cnt"),
    how="left",
    on=["user_id", "merchant_id"],
)
train_data = train_data.merge(
    user_log[["user_id", "merchant_id", "item_id"]]
    .drop_duplicates()
    .groupby(["user_id", "merchant_id"])["item_id"]
    .aggregate("count")
    .rename("um_item_cnt"),
    how="left",
    on=["user_id", "merchant_id"],
)
train_data = train_data.merge(
    user_log[["user_id", "merchant_id", "time_stamp"]]
    .drop_duplicates()
    .groupby(["user_id", "merchant_id"])["time_stamp"]
    .aggregate("std")
    .rename("um_time_std"),
    how="left",
    on=["user_id", "merchant_id"],
)
test_data = test_data.merge(
    user_log[["user_id", "merchant_id", "cat_id"]]
    .drop_duplicates()
    .groupby(["user_id", "merchant_id"])["cat_id"]
    .aggregate("count")
    .rename("um_cat_cnt"),
    how="left",
    on=["user_id", "merchant_id"],
)
test_data = test_data.merge(
    user_log[["user_id", "merchant_id", "item_id"]]
    .drop_duplicates()
    .groupby(["user_id", "merchant_id"])["item_id"]
    .aggregate("count")
    .rename("um_item_cnt"),
    how="left",
    on=["user_id", "merchant_id"],
)
test_data = test_data.merge(
    user_log[["user_id", "merchant_id", "time_stamp"]]
    .drop_duplicates()
    .groupby(["user_id", "merchant_id"])["time_stamp"]
    .aggregate("std")
    .rename("um_time_std"),
    how="left",
    on=["user_id", "merchant_id"],
)


# 后面补的几组特征

前面的统计特征能跑出一个基本结果，下面这些是后来一点点加上去的：复购共现、近期窗口、目标编码、商品和品牌相似度。


## 复购共现

只看发生过复购的用户-商户关系，再做一组 SVD。这个特征比较贴题，能补一些“复购倾向”的信息。


In [ ]:
user_feat_rebuy = pd.DataFrame(
    user_log.user_id.drop_duplicates().sort_values().reset_index(drop=True)
)
user_feat_rebuy = user_feat_rebuy.merge(
    buy.loc[buy["if_rebuy"] == 1, ["user_id", "merchant_id"]]
    .drop_duplicates()
    .groupby("user_id")["merchant_id"]
    .aggregate(merchant_cat),
    how="left",
    on="user_id",
).fillna("0")
user_feat_rebuy


In [ ]:
merchant_cnt = CountVectorizer(
    token_pattern=r"[0-9]+", vocabulary=dict(zip(map(str, list(range(1, 4996))), range(4995)))
)
merchant_cnt_mat = merchant_cnt.fit_transform(user_feat_rebuy.merchant_id.tolist())
svd = TruncatedSVD(n_components=200)
user_mer_feat_mat = svd.fit_transform(merchant_cnt_mat)
print(svd.explained_variance_ratio_.sum())
merchant_feat_rebuy = pd.DataFrame(svd.components_.T)
merchant_feat_rebuy = merchant_feat_rebuy.rename(
    columns=dict(
        zip(
            merchant_feat_rebuy.columns, map(muf_col_rename_f, merchant_feat_rebuy.columns.tolist())
        )
    )
)
mer_user_feat_id_col(merchant_feat_rebuy, merchant_cnt.vocabulary_)
merchant_feat_rebuy["merchant_id"] = merchant_feat_rebuy.merchant_id.astype("int16")
user_feat_rebuy = pd.concat(
    [
        user_feat_rebuy,
        pd.DataFrame(user_mer_feat_mat).rename(
            columns=dict(zip(range(200), map(umf_col_rename_f, list(range(200)))))
        ),
    ],
    axis=1,
).drop("merchant_id", axis=1)


In [ ]:
# 复购共现的点积相似度
add_dot_feature(train_data, user_feat_rebuy, merchant_feat_rebuy, prefix="rebuy")
add_dot_feature(test_data, user_feat_rebuy, merchant_feat_rebuy, prefix="rebuy")

# 复购共现的余弦相似度
add_cos_feature(train_data, user_feat_rebuy, merchant_feat_rebuy, prefix="rebuy")
add_cos_feature(test_data, user_feat_rebuy, merchant_feat_rebuy, prefix="rebuy")


In [ ]:
# 复购商户向量准备并回样本表
merchant_feat_rebuy = merchant_feat_rebuy.drop_duplicates("merchant_id", keep="last").copy()
merchant_feat_rebuy["merchant_id"] = merchant_feat_rebuy["merchant_id"].astype("int32")
train_data["merchant_id"] = train_data["merchant_id"].astype("int32")
test_data["merchant_id"] = test_data["merchant_id"].astype("int32")

# 加个 rebuy_ 前缀，后面看列名更清楚
cols = [c for c in merchant_feat_rebuy.columns if c != "merchant_id"]
merchant_feat_rebuy = merchant_feat_rebuy.rename(columns={c: f"rebuy_{c}" for c in cols})

# 重跑时先清掉旧的 rebuy_ 列
train_data = train_data.drop(
    columns=[c for c in train_data.columns if c.startswith("rebuy_")], errors="ignore"
)
test_data = test_data.drop(
    columns=[c for c in test_data.columns if c.startswith("rebuy_")], errors="ignore"
)

# 每个 merchant_id 只对应一行向量
train_data = train_data.merge(merchant_feat_rebuy, on="merchant_id", how="left", validate="m:1")
test_data = test_data.merge(merchant_feat_rebuy, on="merchant_id", how="left", validate="m:1")


## 临近双十一的行为窗口

这个比赛和双十一强相关，所以我把 11 月 4 日到 11 月 10 日单独拿出来看，和更早的行为做对比。


In [ ]:
# 看近期行为占比，以及近期相对历史有没有变强
# 只用 11 月 10 日及之前的日志，别把预测日信息混进去
logs_pre = user_log[user_log["time_stamp"] <= 1110].copy()

# 这里只看加购和购买，再拆成历史/近期两个窗口
tmp = logs_pre[logs_pre["action_type"].isin([1, 2])].copy()
tmp["tbin"] = np.where(tmp["time_stamp"] <= 1103, "b1", "b3")  # b1 更早，b3 更近


def add_time_ratio_feats(base_df, tmp, key_cols, prefix):
    """给某个粒度加上 b1/b3 两段窗口统计，主要看最近一周有没有升温。"""
    g = tmp.groupby(key_cols + ["tbin", "action_type"]).size().rename("cnt").reset_index()
    pv = g.pivot_table(index=key_cols, columns=["tbin", "action_type"], values="cnt", fill_value=0)
    # pivot 后列名会变成多级索引，这里改回普通列名
    new_cols = {}
    for t, a in pv.columns:
        new_cols[(t, a)] = f"{prefix}_{t}_a{int(a)}_cnt"
    pv.columns = [new_cols[c] for c in pv.columns]
    pv = pv.reset_index()

    # 有些窗口/行为可能完全没出现，手动补 0
    for t in ["b1", "b3"]:
        for a in [1, 2]:
            c = f"{prefix}_{t}_a{a}_cnt"
            if c not in pv.columns:
                pv[c] = 0

    # 两个窗口里的加购+购买总量
    pv[f"{prefix}_all_cnt12"] = (
        pv[f"{prefix}_b1_a1_cnt"]
        + pv[f"{prefix}_b1_a2_cnt"]
        + pv[f"{prefix}_b3_a1_cnt"]
        + pv[f"{prefix}_b3_a2_cnt"]
    )

    # 近期窗口占比
    for a in [1, 2]:
        num = f"{prefix}_b3_a{a}_cnt"
        pv[f"{prefix}_b3_a{a}_ratio"] = pv[num] / (pv[f"{prefix}_all_cnt12"] + 1e-6)

    # 近期和历史的比例
    for a in [1, 2]:
        num = f"{prefix}_b3_a{a}_cnt"
        den = f"{prefix}_b1_a{a}_cnt"
        pv[f"{prefix}_b3b1_a{a}_ratio"] = pv[num] / (pv[den] + 1e-6)

    # 拼回原表，缺失就当作没发生过
    out = base_df.merge(pv, on=key_cols, how="left")
    fill_cols = [c for c in out.columns if c.startswith(prefix + "_")]
    out[fill_cols] = out[fill_cols].fillna(0).astype("float32")
    return out


# 用户粒度
train_data = add_time_ratio_feats(train_data, tmp, ["user_id"], "u")
test_data = add_time_ratio_feats(test_data, tmp, ["user_id"], "u")

# 商户粒度
train_data = add_time_ratio_feats(train_data, tmp, ["merchant_id"], "m")
test_data = add_time_ratio_feats(test_data, tmp, ["merchant_id"], "m")

# 用户-商户这一对的粒度
train_data = add_time_ratio_feats(train_data, tmp, ["user_id", "merchant_id"], "pair")
test_data = add_time_ratio_feats(test_data, tmp, ["user_id", "merchant_id"], "pair")
# 这一组时间窗口特征到这里结束


In [ ]:
# 近期购买越靠近双十一，权重越大
def add_decay(logs, keys, action=2, alpha=0.96, out="decay_b3_a2"):
    tmp = logs[(logs["action_type"] == action) & (logs["time_stamp"].between(1104, 1110))].copy()
    tmp["w"] = (alpha ** (1111 - tmp["time_stamp"].astype("int32"))).astype("float32")
    return tmp.groupby(keys)["w"].sum().rename(out).astype("float32").reset_index()


logs_pre = user_log[user_log["time_stamp"] <= 1110][
    ["user_id", "merchant_id", "time_stamp", "action_type"]
].copy()
pair_decay = add_decay(logs_pre, ["user_id", "merchant_id"], out="pair_decay_b3_a2")
user_decay = add_decay(logs_pre, ["user_id"], out="u_decay_b3_a2")
mer_decay = add_decay(logs_pre, ["merchant_id"], out="m_decay_b3_a2")

for name in ("train_data", "test_data"):
    df = locals()[name]
    df = (
        df.merge(pair_decay, on=["user_id", "merchant_id"], how="left")
        .merge(user_decay, on=["user_id"], how="left")
        .merge(mer_decay, on=["merchant_id"], how="left")
    )
    df[["pair_decay_b3_a2", "u_decay_b3_a2", "m_decay_b3_a2"]] = df[
        ["pair_decay_b3_a2", "u_decay_b3_a2", "m_decay_b3_a2"]
    ].fillna(0).astype("float32")
    df["pair_u_decay_share_b3_buy"] = (df["pair_decay_b3_a2"] / (df["u_decay_b3_a2"] + 1e-6)).astype("float32")
    df["pair_m_decay_share_b3_buy"] = (df["pair_decay_b3_a2"] / (df["m_decay_b3_a2"] + 1e-6)).astype("float32")
    locals()[name] = df


In [ ]:
# 近期加购到购买的大致转化情况
for df in (train_data, test_data):
    need = [
        "pair_b3_a1_cnt",
        "pair_b3_a2_cnt",
        "u_b3_a1_cnt",
        "u_b3_a2_cnt",
        "m_b3_a1_cnt",
        "m_b3_a2_cnt",
    ]
    if all(c in df.columns for c in need):
        df["pair_b3_ctr"] = df["pair_b3_a2_cnt"] / (
            df["pair_b3_a1_cnt"] + df["pair_b3_a2_cnt"] + 1e-6
        )
        df["u_b3_ctr"] = df["u_b3_a2_cnt"] / (df["u_b3_a1_cnt"] + df["u_b3_a2_cnt"] + 1e-6)
        df["m_b3_ctr"] = df["m_b3_a2_cnt"] / (df["m_b3_a1_cnt"] + df["m_b3_a2_cnt"] + 1e-6)
        df["pair_over_u_b3_ctr"] = df["pair_b3_ctr"] / (df["u_b3_ctr"] + 1e-6)
        df["pair_over_m_b3_ctr"] = df["pair_b3_ctr"] / (df["m_b3_ctr"] + 1e-6)


In [ ]:
logs_b3 = user_log[user_log["time_stamp"].between(1104, 1110)][
    ["user_id", "merchant_id", "cat_id", "brand_id"]
].copy()

# 近期看过/买过的品类、品牌覆盖面
u_cat = (
    logs_b3.dropna(subset=["cat_id"])
    .drop_duplicates(["user_id", "cat_id"])
    .groupby("user_id")
    .size()
    .rename("u_b3_uni_cat")
    .reset_index()
)
m_cat = (
    logs_b3.dropna(subset=["cat_id"])
    .drop_duplicates(["merchant_id", "cat_id"])
    .groupby("merchant_id")
    .size()
    .rename("m_b3_uni_cat")
    .reset_index()
)
p_cat = (
    logs_b3.dropna(subset=["cat_id"])
    .drop_duplicates(["user_id", "merchant_id", "cat_id"])
    .groupby(["user_id", "merchant_id"])
    .size()
    .rename("pair_b3_uni_cat")
    .reset_index()
)

u_brd = (
    logs_b3.dropna(subset=["brand_id"])
    .drop_duplicates(["user_id", "brand_id"])
    .groupby("user_id")
    .size()
    .rename("u_b3_uni_brand")
    .reset_index()
)
m_brd = (
    logs_b3.dropna(subset=["brand_id"])
    .drop_duplicates(["merchant_id", "brand_id"])
    .groupby("merchant_id")
    .size()
    .rename("m_b3_uni_brand")
    .reset_index()
)
p_brd = (
    logs_b3.dropna(subset=["brand_id"])
    .drop_duplicates(["user_id", "merchant_id", "brand_id"])
    .groupby(["user_id", "merchant_id"])
    .size()
    .rename("pair_b3_uni_brand")
    .reset_index()
)

for name in ("train_data", "test_data"):
    df = locals()[name]
    df = (
        df.merge(p_cat, on=["user_id", "merchant_id"], how="left")
        .merge(u_cat, on="user_id", how="left")
        .merge(m_cat, on="merchant_id", how="left")
    )
    df = (
        df.merge(p_brd, on=["user_id", "merchant_id"], how="left")
        .merge(u_brd, on="user_id", how="left")
        .merge(m_brd, on="merchant_id", how="left")
    )
    for c in [
        "pair_b3_uni_cat",
        "u_b3_uni_cat",
        "m_b3_uni_cat",
        "pair_b3_uni_brand",
        "u_b3_uni_brand",
        "m_b3_uni_brand",
    ]:
        df[c] = df[c].fillna(0).astype("float32")
    # 用交集/并集粗略看用户和商户的品类、品牌是否匹配
    inter_c = df["pair_b3_uni_cat"]
    union_c = df["u_b3_uni_cat"] + df["m_b3_uni_cat"] - inter_c + 1e-6
    inter_b = df["pair_b3_uni_brand"]
    union_b = df["u_b3_uni_brand"] + df["m_b3_uni_brand"] - inter_b + 1e-6
    df["jac_cat_b3"] = inter_c / union_c
    df["jac_brand_b3"] = inter_b / union_b
    locals()[name] = df


In [ ]:
# 重跑这个单元格前，先把旧的 gap 列删掉
for df in (train_data, test_data):
    df.drop(
        columns=[c for c in df.columns if c.endswith("_gap_buy") or c.endswith("_gap_cart")],
        inplace=True,
        errors="ignore",
    )

logs_pre = user_log[user_log["time_stamp"] <= 1110][
    ["user_id", "merchant_id", "time_stamp", "action_type"]
].copy()
T0 = 1111  # 双十一作为预测基准日


def last_gap(tbl, keys, action, out):
    tmp = tbl[tbl["action_type"] == action].groupby(keys)["time_stamp"].max().reset_index()
    tmp[out] = (T0 - tmp["time_stamp"].astype("int32")).clip(lower=1)
    return tmp[keys + [out]]


pair_buy = last_gap(logs_pre, ["user_id", "merchant_id"], 2, "pair_gap_buy")
pair_cart = last_gap(logs_pre, ["user_id", "merchant_id"], 1, "pair_gap_cart")
user_buy = last_gap(logs_pre, ["user_id"], 2, "u_gap_buy")
mer_buy = last_gap(logs_pre, ["merchant_id"], 2, "m_gap_buy")

for name in ("train_data", "test_data"):
    df = locals()[name]
    df = (
        df.merge(pair_buy, on=["user_id", "merchant_id"], how="left")
        .merge(pair_cart, on=["user_id", "merchant_id"], how="left")
        .merge(user_buy, on=["user_id"], how="left")
        .merge(mer_buy, on=["merchant_id"], how="left")
    )
    for c in ["pair_gap_buy", "pair_gap_cart", "u_gap_buy", "m_gap_buy"]:
        df[c] = df[c].fillna(1e3).astype("float32")  # 没有历史行为就给一个很大的间隔
    locals()[name] = df


## 目标编码

这里做商户和用户的 OOF 目标编码。训练集用折外统计，测试集用全量训练集统计，避免直接把标签泄漏进去。


In [ ]:
# 商户目标编码，用 OOF 防止直接吃到标签
def merchant_oof_te(
    train, test, key="merchant_id", y="label", user="user_id", k=5, m=100, seed=2022, add_logit=True
):
    # 类型先对齐，不然 map 时容易全是缺失
    train[key] = train[key].astype("int32", copy=False)
    test[key] = test[key].astype("int32", copy=False)
    p0 = float(train[y].mean())

    # 尽量按用户分组切折，减少同一用户信息串到验证集
    try:
        splitter = StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=seed)
        splits = splitter.split(
            train[[key]], train[y].astype("int32").values, groups=train[user].values
        )
    except Exception:
        try:
            splitter = GroupKFold(n_splits=k)
            splits = splitter.split(train[[key]], groups=train[user].values)
        except Exception:
            splitter = StratifiedKFold(n_splits=k, shuffle=True, random_state=seed)
            splits = splitter.split(train[[key]], train[y].astype("int32").values)

    oof = pd.Series(np.nan, index=train.index, dtype="float32")
    for tr_idx, va_idx in splits:
        g = train.iloc[tr_idx].groupby(key)[y].agg(["sum", "count"])
        lpe = (g["sum"] + m * p0) / (g["count"] + m)
        oof.iloc[va_idx] = train.iloc[va_idx][key].map(lpe).fillna(p0).astype("float32")

    # 测试集没有标签，只能用完整训练集的平滑统计
    g_full = train.groupby(key)[y].agg(["sum", "count"])
    lpe_full = ((g_full["sum"] + m * p0) / (g_full["count"] + m)).astype("float32")

    te_col = f"{key}_lpe"
    train[te_col] = oof.fillna(p0).astype("float32")
    test[te_col] = test[key].map(lpe_full).fillna(p0).astype("float32")

    if add_logit:
        eps = 1e-6
        for df in (train, test):
            p = df[te_col].clip(eps, 1 - eps).astype("float32")
            df[te_col + "_logit"] = np.log(p / (1 - p)).astype("float32")
    return train, test


# 交互特征做完后，再把商户编码加进去
train_data, test_data = merchant_oof_te(
    train_data, test_data, k=5, m=100, seed=2022, add_logit=True
)


In [ ]:
# 重跑时先删旧的 user_lpe 列
for df in (train_data, test_data):
    df.drop(
        columns=[c for c in df.columns if c.startswith("user_lpe")], inplace=True, errors="ignore"
    )


def oof_te_user(train, test, y="label", group="user_id", k=5, m=200, seed=2022, out="user_lpe"):
    p0 = float(train[y].mean())
    for c in (group,):
        if c in train.columns:
            train[c] = train[c].astype("int32", copy=False)
        if c in test.columns:
            test[c] = test[c].astype("int32", copy=False)
    # 用户目标编码也按用户分组切折
    try:
        splits = StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=seed).split(
            train[[group]], train[y].astype("int32").values, groups=train[group].values
        )
    except Exception:
        splits = GroupKFold(n_splits=k).split(train[[group]], groups=train[group].values)

    oof = np.full(len(train), np.nan, dtype="float32")
    for tr_idx, va_idx in splits:
        g = train.iloc[tr_idx].groupby(group)[y].agg(["sum", "count"])
        te = (g["sum"] + m * p0) / (g["count"] + m)
        oof[va_idx] = train.iloc[va_idx][group].map(te).fillna(p0).astype("float32")
    train[out] = pd.Series(oof, index=train.index).fillna(p0).astype("float32")

    g_full = train.groupby(group)[y].agg(["sum", "count"])
    te_full = ((g_full["sum"] + m * p0) / (g_full["count"] + m)).astype("float32")
    test[out] = test[group].map(te_full).fillna(p0).astype("float32")

    # 顺手加一个 logit 版本
    eps = 1e-6
    for df in (train, test):
        p = df[out].clip(eps, 1 - eps)
        df[out + "_logit"] = np.log(p / (1 - p)).astype("float32")
    return train, test


train_data, test_data = oof_te_user(train_data, test_data, m=200, seed=2022)


## 商品和品牌相似度

除了商户 ID，本身买过/看过的商品和品牌也能表达偏好。这里用 TF-IDF + SVD 做成相似度特征。


In [ ]:
# 商品序列做 TF-IDF/SVD，只留下点积和余弦两个特征
# 只用 11 月 10 日及之前的日志，别把预测日信息混进去
logs_pre = user_log[user_log["time_stamp"] <= 1110].copy()
logs_pre = logs_pre[["user_id", "merchant_id", "item_id"]].dropna()
logs_pre["user_id"] = logs_pre["user_id"].astype("int32")
logs_pre["merchant_id"] = logs_pre["merchant_id"].astype("int32")
logs_pre["item_id"] = logs_pre["item_id"].astype("int32", copy=False)

# 用户侧、商户侧各自拼商品 token
u_bag = logs_pre.groupby("user_id")["item_id"].apply(lambda s: " ".join(map(str, s)))
m_bag = logs_pre.groupby("merchant_id")["item_id"].apply(lambda s: " ".join(map(str, s)))

max_features = 8000
n_components = 64
tfidf_u = TfidfVectorizer(token_pattern=r"\d+", max_features=max_features, dtype=np.float32)
U_tf = tfidf_u.fit_transform(u_bag.values)  # 用户侧

# 商户侧复用用户侧词表，这样两个矩阵在同一个空间里
tfidf_m = TfidfVectorizer(token_pattern=r"\d+", vocabulary=tfidf_u.vocabulary_, dtype=np.float32)
M_tf = tfidf_m.fit_transform(m_bag.values)  # 商户侧

# SVD 在用户侧拟合，然后同一个投影用到商户侧
svd = TruncatedSVD(n_components=n_components, random_state=2022)
U_z = svd.fit_transform(U_tf).astype("float32")  # 用户嵌入
M_z = svd.transform(M_tf).astype("float32")  # 商户嵌入

# 转成 DataFrame，后面按主键 merge 比较方便
u_cols = [f"user_item_svd_{i}" for i in range(n_components)]
m_cols = [f"merch_item_svd_{i}" for i in range(n_components)]
user_emb = pd.DataFrame(U_z, columns=u_cols)
user_emb.insert(0, "user_id", u_bag.index.values.astype("int32"))
mer_emb = pd.DataFrame(M_z, columns=m_cols)
mer_emb.insert(0, "merchant_id", m_bag.index.values.astype("int32"))


# 合并后直接批量算点积和余弦
def add_dot_cos(df, user_emb, mer_emb, u_cols, m_cols, prefix="tfidf_item"):
    out = df.merge(user_emb, on="user_id", how="left").merge(mer_emb, on="merchant_id", how="left")

    U = np.nan_to_num(out[u_cols].to_numpy(dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    V = np.nan_to_num(out[m_cols].to_numpy(dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    dot = (U * V).sum(axis=1).astype("float32")
    un = np.linalg.norm(U, axis=1)
    vn = np.linalg.norm(V, axis=1)
    denom = un * vn
    cos = np.divide((U * V).sum(axis=1), denom, out=np.zeros_like(denom, dtype=np.float32), where=denom > 0)

    out[f"{prefix}_dot"] = dot
    out[f"{prefix}_cos"] = cos.astype("float32")

    # 高维向量不直接留下，只保留两个相似度
    out.drop(columns=u_cols + m_cols, inplace=True, errors="ignore")
    return out


train_data = add_dot_cos(train_data, user_emb, mer_emb, u_cols, m_cols, prefix="tfidf_item")
test_data = add_dot_cos(test_data, user_emb, mer_emb, u_cols, m_cols, prefix="tfidf_item")

# 中间结果用完就删
del logs_pre, U_tf, M_tf, U_z, M_z, user_emb, mer_emb
gc.collect()
# 商品相似度特征到这里结束


In [ ]:
# 品牌也做一组 TF-IDF/SVD，这里只留余弦
# 只看预测日前的加购和购买，浏览日志噪声有点大
b_logs = user_log[(user_log["time_stamp"] <= 1110) & (user_log["action_type"].isin([1, 2]))]
b_logs = b_logs[["user_id", "merchant_id", "brand_id"]].dropna()
b_logs["user_id"] = b_logs["user_id"].astype("int32")
b_logs["merchant_id"] = b_logs["merchant_id"].astype("int32")
b_logs["brand_id"] = b_logs["brand_id"].astype("int32", copy=False)

# 用户侧、商户侧各自拼品牌 token
u_bag_b = b_logs.groupby("user_id")["brand_id"].apply(lambda s: " ".join(map(str, s)))
m_bag_b = b_logs.groupby("merchant_id")["brand_id"].apply(lambda s: " ".join(map(str, s)))

# 词表、min_df 和 sublinear_tf 都是为了压一点噪声
tfidf = TfidfVectorizer(token_pattern=r"\d+", max_features=3000, min_df=3, sublinear_tf=True, dtype=np.float32)
U_tf = tfidf.fit_transform(u_bag_b.values)
M_tf = TfidfVectorizer(token_pattern=r"\d+", vocabulary=tfidf.vocabulary_, dtype=np.float32).fit_transform(
    m_bag_b.values
)

# 品牌偏好做成 32 维表示
svd = TruncatedSVD(n_components=32, random_state=2022)
U = svd.fit_transform(U_tf).astype("float32")
V = svd.transform(M_tf).astype("float32")

# 转成 DataFrame，后面按主键 merge 比较方便
u_cols = [f"u_brand_{i}" for i in range(32)]
m_cols = [f"m_brand_{i}" for i in range(32)]
Udf = pd.DataFrame(U, columns=u_cols)
Udf.insert(0, "user_id", u_bag_b.index.values.astype("int32"))
Mdf = pd.DataFrame(V, columns=m_cols)
Mdf.insert(0, "merchant_id", m_bag_b.index.values.astype("int32"))


# 合并后只留下品牌余弦相似度
def add_brand_cos_only(df, Udf, Mdf, u_cols, m_cols, out_col="tfidf_brand_cos"):
    # 重跑时先删旧的品牌特征
    df.drop(
        columns=[c for c in df.columns if c.startswith("tfidf_brand_")],
        inplace=True,
        errors="ignore",
    )
    out = df.merge(Udf, on="user_id", how="left").merge(Mdf, on="merchant_id", how="left")
    Ux = np.nan_to_num(out[u_cols].to_numpy(np.float32))
    Vx = np.nan_to_num(out[m_cols].to_numpy(np.float32))
    un = np.linalg.norm(Ux, axis=1)
    vn = np.linalg.norm(Vx, axis=1)
    denom = un * vn
    cos = np.divide((Ux * Vx).sum(axis=1), denom, out=np.zeros_like(un), where=denom > 0)
    out[out_col] = cos.astype("float32")
    # 临时向量不保留
    out.drop(columns=u_cols + m_cols, inplace=True, errors="ignore")
    return out


train_data = add_brand_cos_only(train_data, Udf, Mdf, u_cols, m_cols, out_col="tfidf_brand_cos")
test_data = add_brand_cos_only(test_data, Udf, Mdf, u_cols, m_cols, out_col="tfidf_brand_cos")

del b_logs, u_bag_b, m_bag_b, U_tf, M_tf, U, V, Udf, Mdf
gc.collect()
# 品牌相似度特征到这里结束


## 压一下数据类型

特征做完以后列数比较多，训练前把能压的数值类型压一下，省一点内存。


In [ ]:
def shrink(df, exclude=("user_id", "merchant_id", "label")):
    for c in df.columns:
        if c in exclude:
            continue
        if pd.api.types.is_float_dtype(df[c]):
            df[c] = pd.to_numeric(df[c], downcast="float")
        elif pd.api.types.is_integer_dtype(df[c]):
            df[c] = pd.to_numeric(df[c], downcast="integer")
    return df


train_data = shrink(train_data)
test_data = shrink(test_data)


# 训练和提交

最后用 LightGBM 做五折训练，把测试集五折概率平均后生成提交文件。


In [ ]:
def start_train(train, test, ycol, k_fold=5, mode="pred", param_list=None, seed=2025):
    # 先定好训练和测试使用的列
    X_cols = [c for c in train.columns if c != ycol]
    # 测试集少了训练列就补 0
    for c in X_cols:
        if c not in test.columns:
            test[c] = 0

    # float 列统一压到 float32，省点内存
    for df in (train, test):
        for c in X_cols:
            if pd.api.types.is_float_dtype(df[c]):
                df[c] = df[c].astype("float32", copy=False)

    # LightGBM 前统一处理 NaN/inf
    X = train[X_cols].to_numpy(dtype=np.float32, copy=True)
    T = test[X_cols].to_numpy(dtype=np.float32, copy=True)
    np.nan_to_num(X, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
    np.nan_to_num(T, copy=False, nan=0.0, posinf=0.0, neginf=0.0)

    y = train[ycol].to_numpy()

    skf = StratifiedKFold(n_splits=k_fold, shuffle=True, random_state=seed)

    # 每折预测概率累加到这里
    pred = np.zeros((T.shape[0], 2), dtype=np.float64)
    auc = 0.0

    # 这组参数是最后提交用的版本
    if param_list is None:
        params = dict(
            n_estimators=4000,
            learning_rate=0.05,
            num_leaves=32,
            colsample_bytree=0.8,
            subsample=0.9,
            max_depth=7,
            reg_alpha=0.3,
            reg_lambda=0.3,
            min_child_samples=80,
            random_state=seed,
            n_jobs=-1,
        )
    else:
        (
            n_estimators,
            lr,
            num_leaves,
            colsample_bytree,
            subsample,
            max_depth,
            reg_alpha,
            reg_lambda,
        ) = param_list
        params = dict(
            n_estimators=n_estimators,
            learning_rate=lr,
            num_leaves=num_leaves,
            colsample_bytree=colsample_bytree,
            subsample=subsample,
            max_depth=max_depth,
            reg_alpha=reg_alpha,
            reg_lambda=reg_lambda,
            min_child_samples=80,
            random_state=seed,
            n_jobs=-1,
        )

    epoch = 0
    for tr_idx, va_idx in skf.split(X, y):
        epoch += 1
        X_train, X_val = X[tr_idx], X[va_idx]
        y_train, y_val = y[tr_idx], y[va_idx]

        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_train,
            y_train,
            eval_set=[(X_train, y_train), (X_val, y_val)],
            eval_metric="auc",
            callbacks=[early_stopping(40), log_evaluation(100)],
        )

        pred_val = model.predict_proba(X_val)
        print(f"fold {epoch}, val roc-auc: {roc_auc_score(y_val, pred_val[:, 1]):.4f}")
        auc += roc_auc_score(y_val, pred_val[:, 1]) / k_fold

        # 五折结果取平均
        pred += model.predict_proba(T) / k_fold

        del model, X_train, X_val, y_train, y_val
        gc.collect()

    print(f"val mean auc is {auc:.4f}")
    return auc if mode == "eval" else pred


In [ ]:
prob = start_train(train_data, test_data, "label", k_fold=5)


In [ ]:
submit = pd.read_csv(DATA_DIR / "test_format1.csv")  # 提交文件需要保留原来的主键顺序

# start_train 返回的是两列概率，这里取正类概率
prob1 = (
    prob[:, 1]
    if getattr(prob, "ndim", 1) == 2 and prob.shape[1] > 1
    else np.asarray(prob).reshape(-1)
)

# 行数对不上就别直接保存，先报错
assert len(submit) == len(prob1), f"长度不一致: submit={len(submit)}, prob={len(prob1)}"

submit["prob"] = prob1.astype("float32")
submit.to_csv(SUBMISSION_DIR / "submission.csv", index=False)
submit
